In [3]:
### Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv("HR.csv")
print("✅ Data loaded successfully!\n")

print("Available Columns:")
print(list(df.columns), "\n")
print(df.head(), "\n")

# Get user inputs
target_col = input("Enter the target column name: ")
feature_cols = input("Enter feature column names separated by commas: ").split(',')
feature_cols = [col.strip() for col in feature_cols]

X = df[feature_cols].copy()
y = df[target_col]

# ----------------------------------------------
# ✅ Label Encode categorical columns first
# ----------------------------------------------
le = LabelEncoder()

for col in ['salary', 'Department', 'department']:
    if col in X.columns:
        X[col] = le.fit_transform(X[col])

# ----------------------------------------------
# ✅ OneHotEncode categorical columns safely
# ----------------------------------------------
cat_cols = [col for col in ['salary', 'Department', 'department'] if col in X.columns]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Split the dataset
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
    )
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

# ----------------------------------------------
# ✅ Ask which model to train
# ----------------------------------------------
model_type = input("Which model do you want to use? (linear/logistic): ").lower()

if model_type == "linear":
    reg = LinearRegression()
    reg.fit(X_train, y_train)
    print("\n✅ Linear Regression Model Trained Successfully!")

elif model_type == "logistic":
    reg = LogisticRegression(max_iter=1000)
    reg.fit(X_train, y_train)
    print("\n✅ Logistic Regression Model Trained Successfully!")

else:
    print("❌ Invalid model name!")
    exit()

print("Model Score:", reg.score(X_test, y_test))

# ----------------------------------------------
# ✅ Prediction with user input
# ----------------------------------------------
print("\n--- Enter Employee Data for Prediction ---")
new_data = {}

for col in feature_cols:
    if col.lower() in ['salary']:
        val = input("Enter Salary (Low / Medium / High): ").strip().lower()
        new_data[col] = val
    elif col.lower() in ['department', 'Department']:
        val = input(f"Enter Department (choose from {df['Department'].unique().tolist() if 'Department' in df.columns else df['department'].unique().tolist()}): ").strip()
        new_data[col] = val
    else:
        val = float(input(f"Enter numeric value for {col}: "))
        new_data[col] = val

new_df = pd.DataFrame([new_data])

# Encode the new input using the same method
for col in ['salary', 'Department', 'department']:
    if col in new_df.columns:
        new_df[col] = le.fit_transform(new_df[col])

# OneHotEncode the new input and align with training columns
new_df = pd.get_dummies(new_df, columns=cat_cols, drop_first=True)
new_df = new_df.reindex(columns=X.columns, fill_value=0)

# Predict
pred = reg.predict(new_df)

print("\n🎯 Prediction for this employee:", pred)
if model_type == "logistic":
    print("Explanation: 1 = Will Leave, 0 = Will Stay")


✅ Data loaded successfully!

Available Columns:
['satisfaction_level', 'last_evaluation', 'number_project', 'average_montly_hours', 'time_spend_company', 'Work_accident', 'left', 'promotion_last_5years', 'Department', 'salary'] 

   satisfaction_level  last_evaluation  number_project  average_montly_hours  \
0                0.38             0.53               2                   157   
1                0.80             0.86               5                   262   
2                0.11             0.88               7                   272   
3                0.72             0.87               5                   223   
4                0.37             0.52               2                   159   

   time_spend_company  Work_accident  left  promotion_last_5years Department  \
0                   3              0     1                      0      sales   
1                   6              0     1                      0      sales   
2                   4              0     1       

Enter the target column name:  left
Enter feature column names separated by commas:  satisfaction_level,Department,salary
Which model do you want to use? (linear/logistic):  logistic



✅ Logistic Regression Model Trained Successfully!
Model Score: 0.7816666666666666

--- Enter Employee Data for Prediction ---


Enter numeric value for satisfaction_level:  0.39
Enter Department (choose from ['sales', 'accounting', 'hr', 'technical', 'support', 'management', 'IT', 'product_mng', 'marketing', 'RandD']):  Sales
Enter Salary (Low / Medium / High):  Low



🎯 Prediction for this employee: [0]
Explanation: 1 = Will Leave, 0 = Will Stay
